# 1-B：Agent，路線由 AI 決定；tool 與 MCP

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AluminumShark/langgraph-skills-workshop/blob/main/notebooks/1-B_agent.ipynb)

> 📍 **對應簡報**：第 1 堂「第三站：Agent 1-B」，投影片 **30–39**；核心迴圈選做在附錄 **47**（課堂時間 1:10–1:40）

## 這本 notebook 在做什麼

1-A 的小助教，步驟、分支與停止條件都由程式安排（workflow）。這本換個做法：**不畫流程圖，只給 AI 一個工具箱**，
告訴它有哪些工具、目標是什麼，讓它自己決定先用哪個、用幾次。這種做法叫 **agent**：

> **Agent ＝ 模型 ＋ 工具 ＋ 迴圈**：問模型 → 它說要用哪個工具，你的程式就去執行、把結果貼回去 → 再問模型 → 直到它說「不用工具了」。
> 模型本身不會執行程式，它只會「說」想用哪個工具；真正做事的是你的程式（細節在 1-0）。

工具箱裡有三把自己寫的工具（第 6 段還會從 MCP 插座多接幾把）：

| 工具 | 做什麼 |
| --- | --- |
| `run_code` | 在教學限制下執行作業，回傳輸出或錯誤 |
| `check_indent` | 檢查縮排和每行長度（故意不檢查命名，留給第 2 堂） |
| `read_spec` | 讀這一題的作業要求 |

| 段落 | 你會做什麼 | 對應投影片 |
| --- | --- | --- |
| 第 1 段 | 準備三個工具，測試 run_code 的限制與錯誤訊息 | 31–32 |
| 第 2 段 | 查看模型拿到的工具說明書 | 33 |
| 第 3 段 | 建立本地 agent，觀察工具執行紀錄 | 34、35 |
| 第 4 段 | 跑三次比較，再調整 system prompt | 36 |
| 第 5 段 | 區分自行停止、達到步數上限與成功 | 37–38 |
| 第 6 段（教師示範） | 接入 MCP 工具；工具說明長度比較選讀 | 39 |
| 第 7 段（課後選做） | 用 LangGraph 重建核心迴圈 | 47 |

## 怎麼用

1. 先執行 **「0. 環境設定」**。
2. 要改的只有表單欄位：**作業**、**system prompt 加的一句話**（第 4 段）。
3. 每一段：**🤔 先寫預測 → ▶ 執行 → 👀 對照**；核心依序完成第 1–5 段；第 6 段跟著教師看，第 7 段課後選做。
4. 斷線後先重跑環境設定，以及工具與 `agent.py` 的寫檔格，再回到要做的段落。寫檔格只需執行，不必逐行讀。

## 🎯 這兩堂課的小專案：作業小助教

**作業小助教**是一個協助檢查 Python 作業的小程式。你交給它一段作業，它回覆檢查結果。
在 workflow 版本中，語法有錯會先回覆錯誤；語法通過才請 AI 產生含**「優點：」與「改進：」**的回饋。
格式檢查若一直不通過，流程會在達到生成次數上限時停止，並標示未通過，不能把「停止」當成「合格」。

**先分清三件事：**語法正確（Python 看得懂）≠ 執行成功（跑的時候沒出錯）≠ 答案正確（符合題目要求）。

選它當練習，是因為它剛好三種東西都用得到：

| 小助教需要的 | 例子 | 我們用什麼做 |
| --- | --- | --- |
| 由流程控制的步驟 | 先檢查語法；有錯就回報，通過才產生回饋並檢查格式 | 用程式定義**流程**（LangGraph，第 1 堂） |
| 需要判斷的事 | 回饋怎麼寫、要不要真的執行程式看看 | 交給 **AI**（agent，第 1 堂） |
| 做法和規矩 | 回饋的語氣要溫和還是嚴格、好的變數名稱長什麼樣 | 寫成給 AI 看的**手冊**（Skill，第 2 堂） |

👉 **這本 notebook 做的是：**讓 AI 自己決定怎麼批改的小助教（agent）：不畫流程圖，只給它一個工具箱（執行作業、檢查縮排、讀題目，再加上從 MCP 插座接進來的「作業倉庫」），看它自己選工具，也看它會出什麼狀況。

<details><summary>想查零件名稱時再展開（不用先背）</summary>

| 零件 | 白話說它在做什麼 |
| --- | --- |
| `check_syntax`（檢查語法） | 只看程式的「文法」對不對，例如括號有沒有關好、冒號有沒有漏；**不會執行程式** |
| `explain_error`（解釋錯誤） | 語法錯的時候，用一句話告訴學生錯在第幾行 |
| `give_feedback`（寫回饋） | 在 1-A 接上模型後，請 AI 寫出「優點＋改進」回饋；是否符合格式由下一站檢查 |
| `check_format`（檢查格式） | 檢查兩個標記的順序、內容是否非空與總字數；不驗證回饋內容是否正確 |
| 工具：`run_code`、`check_indent`、`read_spec`、`check_naming` | 真的執行作業、檢查縮排、讀題目要求、檢查變數名稱 |
| 手冊：`friendly-feedback`、`strict-review` | 告訴 AI 回饋要溫和鼓勵，還是嚴格審查 |

</details>

**操作提醒：**先執行「0. 環境設定」，再照本段的準備格 → 預測 → 執行 → 對照操作。
`%%writefile` 表示「把這格內容存成檔案」；修改後要執行才會生效。重跑準備格可能覆寫手冊，先存好自己的修改。

In [ ]:
#@title 0. 環境設定（第一次打開、或執行階段重設後執行）
#@markdown **模型來源**：`colab` ＝ Colab 內建模型（不另填 API key；可用性與額度依帳號）；`gemini` ＝ 講師提供的 Gemini API key（備案）
模型來源 = "colab"  #@param ["colab", "gemini"]
#@markdown 選 `gemini` 時：建議在左邊 🔑「密鑰」新增 `GOOGLE_API_KEY`；也可以把 key 貼在下面（⚠️ 貼了就不要把 notebook 分享出去）
講師_API_KEY = ""  #@param {type:"string"}
#@markdown 進階（通常不用改）：指定模型名稱，留空用預設
colab_模型名稱 = ""  #@param {type:"string"}
gemini_模型名稱 = "gemini-flash-latest"  #@param {type:"string"}

import os, shutil, subprocess, sys
from pathlib import Path

# 1. 用 uv 安裝套件（限制相容版本範圍；時間依網路而異）
if not os.environ.get("WORKSHOP_SKIP_INSTALL"):
    print("⏳ 用 uv 安裝 LangGraph / LangChain ...")
    if shutil.which("uv") is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(f'uv pip install --python "{sys.executable}" -q "langgraph>=1.1,<2" "langchain>=1.2,<2" "langchain-google-genai>=4,<5" "langgraph-checkpoint-sqlite>=3,<4" "mcp>=1.20,<2" "langchain-mcp-adapters>=0.3,<0.4"', shell=True, check=True)

# 2. 寫出這本 notebook 用到的共用檔案
RB = Path("/content/review-bot")
for d in ['tools', 'deploy', 'skills/friendly-feedback', 'skills/strict-review', 'skills/naming-check/scripts', 'skills/detailed-feedback/scripts']:
    (RB / d).mkdir(parents=True, exist_ok=True)
_檔案 = {
    'llm.py': r'''"""模型轉接頭：把「Colab 內建模型」或「講師的 Gemini API key」包成同一個 model 物件。

學生不用看這個檔案。後面所有 cell 都只寫 model.invoke(...)、model.bind_tools(...)，
不管背後是哪一家模型，用法都一樣。

- colab ：google.colab.ai.generate_text（免費帳號可用，不需要 API key）
- gemini：langchain_google_genai.ChatGoogleGenerativeAI（需要 GOOGLE_API_KEY）

generate_text 只吃一段文字、只回一段文字，不支援原生的 tool calling。
所以 ColabAIChat 做了一個小轉接：
1. 把多則訊息接成一段文字（系統指示、使用者、AI、工具結果各標一個標題）
2. 有綁工具時，在文字最後附上工具清單，請模型「需要工具就只回一行 JSON」
3. 把模型回的 JSON 解析成 tool_calls，格式和原生 tool calling 一模一樣
這剛好讓你看到：模型從頭到尾只是在「說」它想用哪個工具。
"""

from __future__ import annotations

import json
import os
import re
import time
import uuid
from pathlib import Path
from typing import Any

from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage, BaseMessage, ToolMessage
from langchain_core.outputs import ChatGeneration, ChatResult
from langchain_core.utils.function_calling import convert_to_openai_tool

設定檔 = Path(__file__).with_name(".llm.json")
預設_gemini_模型 = "gemini-flash-latest"


def 估算_token(text: str) -> int:
    """粗估 token 數：中日韓文字約 1 字 1 token，其他字元約 4 個 1 token。"""
    cjk = len(re.findall(r"[⺀-鿿豈-﫿＀-￯]", text))
    return max(1, round(cjk + (len(text) - cjk) / 4))


# ---------- 把訊息接成一段文字 ----------

def _工具說明(tools: list[dict]) -> str:
    lines = []
    for t in tools:
        fn = t.get("function", t)
        props = fn.get("parameters", {}).get("properties", {})
        params = ", ".join(f"{k}: {v.get('type', 'string')}" for k, v in props.items())
        desc = " ".join((fn.get("description") or "").split())
        lines.append(f"- {fn['name']}({params})：{desc}")
    return "\n".join(lines)


def _訊息轉文字(messages: list[BaseMessage], tools: list[dict] | None, tool_choice: Any) -> str:
    parts = []
    for m in messages:
        if m.type == "system":
            parts.append(f"【系統指示】\n{m.text}")
        elif m.type == "human":
            parts.append(f"【使用者】\n{m.text}")
        elif m.type == "ai":
            if m.tool_calls:
                calls = [{"name": c["name"], "args": c["args"]} for c in m.tool_calls]
                parts.append("【AI】\n" + json.dumps({"tool_calls": calls}, ensure_ascii=False))
            else:
                parts.append(f"【AI】\n{m.text}")
        elif m.type == "tool":
            name = getattr(m, "name", None) or "工具"
            parts.append(f"【工具 {name} 的結果】\n{m.text}")
    if tools:
        must = ""
        if tool_choice in ("any", "required"):
            must = "這一次你一定要呼叫至少一個工具。\n"
        elif isinstance(tool_choice, str) and tool_choice not in ("auto", "none"):
            must = f"這一次你一定要呼叫 {tool_choice}。\n"
        parts.append(
            "【你可以用的工具】\n"
            f"{_工具說明(tools)}\n\n"
            "【回覆規則】\n"
            f"{must}"
            "如果你需要使用工具，只回一行 JSON，不要加任何其他文字、也不要加 ``` 符號：\n"
            '{"tool_calls": [{"name": "工具名稱", "args": {"參數名稱": "參數值"}}]}\n'
            "如果不需要工具（例如已經有足夠的資訊），就直接用一般文字回答，不要輸出 JSON。"
        )
    parts.append("【AI】")
    return "\n\n".join(parts)


def _解析回覆(text: str, tools: list[dict] | None) -> AIMessage:
    """模型回的是 JSON 紙條就轉成 tool_calls，否則當成一般文字。"""
    meta = {"raw_text": text, "model_provider": "colab"}
    if tools:
        s = text.strip()
        s = re.sub(r"^```(?:json)?\s*|\s*```$", "", s)
        start, end = s.find("{"), s.rfind("}")
        if start != -1 and end > start and ("tool_calls" in s or s.startswith("{")):
            try:
                data = json.loads(s[start : end + 1])
            except json.JSONDecodeError:
                data = None
            calls = None
            if isinstance(data, dict):
                if isinstance(data.get("tool_calls"), list):
                    calls = data["tool_calls"]
                elif "name" in data or "tool" in data:
                    calls = [data]
            if calls:
                tool_calls = []
                for c in calls:
                    if not isinstance(c, dict):
                        continue
                    name = c.get("name") or c.get("tool")
                    args = c.get("args") or c.get("arguments") or {}
                    if isinstance(args, str):
                        try:
                            args = json.loads(args)
                        except json.JSONDecodeError:
                            args = {}
                    if name:
                        tool_calls.append(
                            {"name": name, "args": args, "id": f"call_{uuid.uuid4().hex[:8]}", "type": "tool_call"}
                        )
                if tool_calls:
                    return AIMessage(content="", tool_calls=tool_calls, response_metadata=meta)
    return AIMessage(content=text, response_metadata=meta)


class ColabAIChat(BaseChatModel):
    """把 google.colab.ai.generate_text 包成 LangChain 的聊天模型。"""

    model_name: str | None = None
    max_retries: int = 3

    @property
    def _llm_type(self) -> str:
        return "colab-ai"

    def bind_tools(self, tools, *, tool_choice=None, **kwargs):
        formatted = [convert_to_openai_tool(t) for t in tools]
        return self.bind(tools=formatted, tool_choice=tool_choice)

    def _generate(self, messages, stop=None, run_manager=None, **kwargs) -> ChatResult:
        from google.colab import ai

        tools = kwargs.get("tools")
        prompt = _訊息轉文字(messages, tools, kwargs.get("tool_choice"))
        extra = {"model_name": self.model_name} if self.model_name else {}
        for attempt in range(self.max_retries):
            try:
                text = ai.generate_text(prompt, **extra) or ""
                break
            except Exception:
                if attempt == self.max_retries - 1:
                    raise
                time.sleep(2 * (attempt + 1))
        message = _解析回覆(text, tools)
        message.usage_metadata = {
            "input_tokens": 估算_token(prompt),
            "output_tokens": 估算_token(text),
            "total_tokens": 估算_token(prompt) + 估算_token(text),
        }
        message.response_metadata["token_count"] = "估算值（Colab 內建模型不回報用量）"
        return ChatResult(generations=[ChatGeneration(message=message)])


# ---------- 對外：設定與取得 model ----------

def 讀設定() -> dict:
    try:
        return json.loads(設定檔.read_text(encoding="utf-8"))
    except (FileNotFoundError, json.JSONDecodeError):
        return {}


def 存設定(provider: str, colab_model: str = "", gemini_model: str = "") -> None:
    """記住選了哪個模型來源（不存 API key）。"""
    設定檔.write_text(
        json.dumps(
            {"provider": provider, "colab_model": colab_model, "gemini_model": gemini_model or 預設_gemini_模型},
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )


def _找_api_key() -> str | None:
    if os.environ.get("GOOGLE_API_KEY"):
        return os.environ["GOOGLE_API_KEY"]
    try:
        from google.colab import userdata

        key = userdata.get("GOOGLE_API_KEY")
        if key:
            os.environ["GOOGLE_API_KEY"] = key
            return key
    except Exception:
        pass
    return None


def get_model() -> BaseChatModel:
    cfg = 讀設定()
    # 上雲端時沒有設定檔：用環境變數 WORKSHOP_PROVIDER=gemini（金鑰從 GOOGLE_API_KEY 讀）
    provider = os.environ.get("WORKSHOP_PROVIDER") or cfg.get("provider", "colab")
    if provider == "gemini":
        key = _找_api_key()
        if not key:
            raise RuntimeError(
                "選了 gemini 但找不到 API key：請在左邊 🔑「密鑰」新增 GOOGLE_API_KEY，"
                "或在「0. 環境設定」的表單貼上講師提供的 key，再重跑那一格。"
            )
        from langchain_google_genai import ChatGoogleGenerativeAI

        return ChatGoogleGenerativeAI(
            model=cfg.get("gemini_model") or 預設_gemini_模型, google_api_key=key, max_retries=3
        )
    try:
        from google.colab import ai  # noqa: F401
    except ImportError as exc:
        raise RuntimeError("google.colab.ai 只能在 Google Colab 上使用；不在 Colab 的話請改選 gemini。") from exc
    return ColabAIChat(model_name=cfg.get("colab_model") or None)
''',
    'workshop.py': r'''"""工作坊 notebook 共用的小工具（學生不用看）。

- 準備()          ：每一段的第一格都會呼叫，重新連上模型、重新載入改過的檔案
- 範例作業         ：表單下拉選單裡的幾段作業
- 記下() / 印出歷程()：保存實驗格指定的結果，最後一格按紀錄標題列出
- 顯示流程圖()      ：把 LangGraph 畫成圖
- 存到雲端硬碟() / 下載壓縮檔()
"""

from __future__ import annotations

import datetime as _dt
import importlib
import json
import shutil
import sys
from pathlib import Path

from langchain_core.callbacks import BaseCallbackHandler
from langchain_core.messages import BaseMessage

from llm import get_model, 估算_token  # noqa: F401  （讓 notebook 可以直接 from workshop import 估算_token）

__all__ = [
    "BASE", "準備", "範例作業", "取得作業", "記下", "印出狀態", "印出訂單紙", "印出歷程", "清除歷程",
    "模型呼叫計數器", "顯示流程圖", "存到雲端硬碟", "下載壓縮檔", "估算_token",
]

BASE = Path(__file__).resolve().parent
歷程檔 = BASE / "history.jsonl"

範例作業 = {
    "少了右括號": "print('hi'\n",
    "正確的程式": "def add(a, b):\n    return a + b\n\nprint(add(2, 3))\n",
    "空白": "",
    "變數打錯字": "def add(a, b):\n    total = a + b\n    return summ\n\nprint(add(2, 3))\n",
    "縮排亂掉": "def average(scores):\n  total = 0\n  for s in scores:\n        total += s\n  return total / len(scores)\n\nprint(average([90, 85, 77]))\n",
    "命名不一致": (
        "def CalcAverage(scoreList):\n"
        "    Total = 0\n"
        "    for s in scoreList:\n"
        "        Total += s\n"
        "    avgScore = Total / len(scoreList)\n"
        "    return avgScore\n"
        "\n"
        "myScores = [90, 85, 77]\n"
        "MAX = 100\n"
        "result_2 = CalcAverage(myScores)\n"
        "print(result_2, '/', MAX)\n"
    ),
}


def 取得作業(名稱: str) -> str:
    """表單選「我的作業」時讀 /content/review-bot/我的作業.py，其他從範例裡拿。"""
    if 名稱.startswith("我的作業"):
        path = BASE / "我的作業.py"
        if not path.exists():
            raise FileNotFoundError("還沒有「我的作業」：請先執行上面寫著 %%writefile ... 我的作業.py 的那一格。")
        return path.read_text(encoding="utf-8")
    return 範例作業[名稱]


def 準備():
    """每一段的起點：把 review-bot 放進 import 路徑、重新載入改過的檔案、回傳 model。"""
    if str(BASE) not in sys.path:
        sys.path.insert(0, str(BASE))
    importlib.invalidate_caches()
    for name, mod in list(sys.modules.items()):
        path = getattr(mod, "__file__", None) or ""
        if path and Path(path).resolve().is_relative_to(BASE) and name not in ("workshop", "llm"):
            try:
                importlib.reload(mod)
            except Exception:
                sys.modules.pop(name, None)
    _讓_mcp_stdio_能在notebook用()
    return get_model()


def _讓_mcp_stdio_能在notebook用() -> None:
    """mcp 的 stdio_client 會把 MCP server 的錯誤輸出接到 sys.stderr。
    Colab（Linux）的 sys.stderr 有真的檔案代號，沒問題；Windows 的 Jupyter 沒有，就改接到 log 檔。"""
    try:
        sys.stderr.fileno()
        return
    except Exception:
        pass
    try:
        import mcp.client.stdio as mcp_stdio
    except ImportError:
        return
    (BASE / "deploy").mkdir(exist_ok=True)
    log = open(BASE / "deploy" / "mcp_server.log", "a", encoding="utf-8")
    mcp_stdio.stdio_client.__wrapped__.__defaults__ = (log,)


# ---------- State 歷程 ----------

def _可存的(obj):
    if isinstance(obj, BaseMessage):
        d = {"type": obj.type, "content": obj.text}
        if getattr(obj, "tool_calls", None):
            d["tool_calls"] = [{"name": c["name"], "args": c["args"]} for c in obj.tool_calls]
        if obj.type == "tool":
            d["name"] = getattr(obj, "name", None)
        return d
    if isinstance(obj, dict):
        return {str(k): _可存的(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_可存的(v) for v in obj]
    if isinstance(obj, (str, int, float, bool)) or obj is None:
        return obj
    return str(obj)


def 記下(標題: str, 內容) -> None:
    """把這一次的結果（通常是 State）存進 history.jsonl。"""
    row = {"時間": _dt.datetime.now().strftime("%H:%M:%S"), "標題": 標題, "內容": _可存的(內容)}
    with 歷程檔.open("a", encoding="utf-8") as f:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")


def _短(text, n=300):
    text = str(text)
    return text if len(text) <= n else text[:n] + f"…（還有 {len(text) - n} 字）"


def 印出狀態(state: dict, 縮排: str = "  ") -> None:
    for k, v in state.items():
        if k == "messages":
            print(f"{縮排}{k}: （{len(v)} 則訊息）")
            for m in v:
                m = _可存的(m)
                if isinstance(m, dict) and m.get("tool_calls"):
                    print(f"{縮排}  [{m['type']}] 呼叫 {m['tool_calls']}")
                elif isinstance(m, dict):
                    print(f"{縮排}  [{m.get('type')}] {_短(m.get('content', ''), 200)}")
        else:
            print(f"{縮排}{k}: {_短(repr(v) if not isinstance(v, str) else repr(v), 300)}")


def 印出訂單紙(第幾步: int, state: dict, 上一步: dict | None = None, 不印: tuple = ("code",)) -> None:
    """stream_mode="values" 每一步都會給整張訂單紙；順便標出這一步新寫了哪幾格。"""
    if not 上一步:
        說明 = "起點：空白訂單紙"
    else:
        新寫入 = [k for k in state if state.get(k) != 上一步.get(k)]
        說明 = "新寫入：" + ("、".join(新寫入) if 新寫入 else "（沒有改變）")
    print(f"第 {第幾步} 步（{說明}）")
    print("  ", {k: (_短(v, 120) if isinstance(v, str) else v) for k, v in state.items() if k not in 不印})


def 印出歷程() -> None:
    if not 歷程檔.exists():
        print("還沒有任何紀錄：先跑上面幾段，再回來執行這一格。")
        return
    rows = [json.loads(line) for line in 歷程檔.read_text(encoding="utf-8").splitlines() if line.strip()]
    print(f"📸 共 {len(rows)} 筆紀錄（存在 {歷程檔}）\n")
    for i, row in enumerate(rows, 1):
        print(f"===== {i}. [{row['時間']}] {row['標題']} =====")
        內容 = row["內容"]
        if isinstance(內容, dict):
            印出狀態(內容)
        elif isinstance(內容, list):
            for j, item in enumerate(內容, 1):
                print(f"  --- 第 {j} 步 ---")
                if isinstance(item, dict):
                    印出狀態(item, "    ")
                else:
                    print(f"    {_短(item)}")
        else:
            print(f"  {_短(內容, 1000)}")
        print()


def 清除歷程() -> None:
    歷程檔.unlink(missing_ok=True)


# ---------- 數模型被呼叫了幾次 ----------

class 模型呼叫計數器(BaseCallbackHandler):
    """放進 config={"callbacks": [計數器]}，每呼叫一次模型就 +1。"""

    def __init__(self):
        self.次數 = 0

    def on_chat_model_start(self, serialized, messages, **kwargs):
        self.次數 += 1

    def on_llm_start(self, serialized, prompts, **kwargs):
        self.次數 += 1


# ---------- 畫流程圖 ----------

def 顯示流程圖(graph) -> None:
    g = graph.get_graph()
    try:
        from IPython.display import Image, display

        display(Image(g.draw_mermaid_png()))
    except Exception:
        print("（畫圖服務暫時連不上，改用文字版流程圖；可以貼到 https://mermaid.live 看圖）\n")
        print(g.draw_mermaid())


# ---------- 保存 ----------

def 存到雲端硬碟() -> None:
    from google.colab import drive

    drive.mount("/content/drive")
    目的地 = Path("/content/drive/MyDrive/langgraph-skills-workshop") / _dt.datetime.now().strftime("%Y%m%d-%H%M%S")
    shutil.copytree(BASE, 目的地, ignore=shutil.ignore_patterns("__pycache__", ".llm.json"))
    print(f"✅ 已存到雲端硬碟：{目的地}")
    print("   （notebook 本身請用上方選單「檔案 → 在雲端硬碟中儲存副本」）")


def 下載壓縮檔() -> None:
    from google.colab import files

    名稱 = BASE.parent / ("review-bot-" + _dt.datetime.now().strftime("%Y%m%d-%H%M%S"))
    zip_path = shutil.make_archive(str(名稱), "zip", BASE.parent, BASE.name)
    print(f"✅ 已打包：{zip_path}")
    files.download(zip_path)
''',
    'tools/__init__.py': r'''"""作業小助教的工具箱：run_code、check_indent、read_spec、check_naming。"""
''',
    'tools/run_code.py': r'''"""run_code：在籠子裡執行學生的作業。

學生作業裡什麼都可能有（import os 刪檔、無限迴圈），所以這個工具有三道圍欄：
1. 執行前先檢查：不准 import os / sys / subprocess / shutil 這類危險模組，
   也不准用 eval、exec、open 和底線開頭的屬性（常被拿來繞過限制）
2. 開一個獨立的子程序執行，超過時間就強制停止（無限迴圈也停得下來）
3. 只回一行結果（第幾行、什麼錯），模型才看得懂

⚠️ 這是教學用的圍欄。正式產品請用真正的 sandbox（容器、RestrictedPython、e2b、Modal）。
"""

import ast
import json
import os
import subprocess
import sys
import tempfile

from langchain.tools import tool

禁止的模組 = {"os", "sys", "subprocess", "shutil", "socket", "pathlib", "ctypes", "importlib",
           "multiprocessing", "threading", "signal", "pickle", "builtins", "io", "inspect", "gc"}
允許的模組 = {"math", "random", "string", "re", "datetime", "collections", "itertools", "functools",
           "statistics", "time", "decimal", "fractions", "json", "copy", "heapq", "bisect"}
禁止的函式 = {"eval", "exec", "compile", "open", "input", "__import__", "globals", "locals", "vars",
           "getattr", "setattr", "delattr", "breakpoint", "help", "exit", "quit", "memoryview"}
禁止的屬性 = {"system", "popen", "modules", "builtins", "f_globals", "f_back", "gi_frame", "cr_frame"}
最長時間_秒 = 3
最長輸出_字 = 800

# 子程序裡真正執行作業的那段程式：用「縮小版的 builtins」執行，import 只放行白名單
_RUNNER = r"""
import builtins, contextlib, io, json, sys, traceback
code = sys.stdin.buffer.read().decode("utf-8")
allowed = set(json.loads(sys.argv[1]))
blocked = set(json.loads(sys.argv[2]))
real_import = builtins.__import__
def guarded_import(name, globals=None, locals=None, fromlist=(), level=0):
    if name.split(".")[0] not in allowed:
        raise ImportError("這個工具不允許 import " + name)
    return real_import(name, globals, locals, fromlist, level)
safe = {k: v for k, v in vars(builtins).items() if k not in blocked}
safe["__import__"] = guarded_import
try:
    import resource
    resource.setrlimit(resource.RLIMIT_AS, (512 * 1024 * 1024, 512 * 1024 * 1024))
except Exception:
    pass
out = io.StringIO()
result = {"ok": True}
try:
    compiled = compile(code, "作業.py", "exec")
    with contextlib.redirect_stdout(out):
        exec(compiled, {"__builtins__": safe, "__name__": "__main__"})
except SyntaxError as e:
    result = {"ok": False, "line": e.lineno, "error": "SyntaxError: " + str(e.msg)}
except BaseException as e:
    line = None
    for frame in traceback.extract_tb(e.__traceback__):
        if frame.filename == "作業.py":
            line = frame.lineno
    result = {"ok": False, "line": line, "error": type(e).__name__ + ": " + str(e)}
result["stdout"] = out.getvalue()
sys.__stdout__.write(json.dumps(result))
"""


def _先檢查(code: str) -> str | None:
    """執行前的靜態檢查；有問題就回一行說明，沒問題回 None。"""
    try:
        tree = ast.parse(code)
    except SyntaxError as e:
        return f"❌ 第 {e.lineno} 行 SyntaxError: {e.msg}"
    for node in ast.walk(tree):
        if isinstance(node, (ast.Import, ast.ImportFrom)):
            names = [a.name for a in node.names] if isinstance(node, ast.Import) else [node.module or ""]
            for name in names:
                root = name.split(".")[0]
                if root in 禁止的模組 or root not in 允許的模組:
                    return f"⛔ 已擋下：第 {node.lineno} 行 import {name}（這個工具不允許使用 {root}）"
        elif isinstance(node, ast.Name) and node.id in 禁止的函式:
            return f"⛔ 已擋下：第 {node.lineno} 行用了 {node.id}（這個工具不允許）"
        elif isinstance(node, ast.Attribute) and (
            node.attr.startswith("_") or node.attr in 禁止的屬性 or node.attr in 禁止的模組
        ):
            return f"⛔ 已擋下：第 {node.lineno} 行存取了 .{node.attr}（這個工具不允許）"
    return None


def 執行作業(code: str, 最長時間: float = 最長時間_秒) -> str:
    """真正做事的函式：檢查 → 子程序執行 → 回傳一行結果。"""
    if not code.strip():
        return "❌ 作業是空白的，沒有東西可以執行"
    擋下 = _先檢查(code)
    if 擋下:
        return 擋下
    env = {"PYTHONIOENCODING": "utf-8", "PATH": os.environ.get("PATH", "")}
    if os.name == "nt":
        env["SYSTEMROOT"] = os.environ.get("SYSTEMROOT", "")
    try:
        proc = subprocess.run(
            [sys.executable, "-I", "-c", _RUNNER, json.dumps(sorted(允許的模組)), json.dumps(sorted(禁止的函式))],
            input=code, capture_output=True, text=True, encoding="utf-8",
            timeout=最長時間, cwd=tempfile.gettempdir(), env=env,
        )
    except subprocess.TimeoutExpired:
        return f"⏱️ 執行超過 {最長時間:g} 秒被強制停止（可能有無限迴圈）"
    try:
        result = json.loads(proc.stdout.strip().splitlines()[-1])
    except (IndexError, json.JSONDecodeError):
        return "❌ 執行失敗（程式沒有正常結束，可能用了太多記憶體）"
    stdout = result.get("stdout", "")
    if len(stdout) > 最長輸出_字:
        stdout = stdout[:最長輸出_字] + "\n…（輸出太長，後面省略）"
    if result["ok"]:
        return f"✅ 執行成功，輸出：\n{stdout}" if stdout else "✅ 執行成功（沒有印出任何東西）"
    where = f"第 {result['line']} 行 " if result.get("line") else ""
    印出 = f"\n（出錯前印出：{stdout.strip()}）" if stdout.strip() else ""
    return f"❌ {where}{result['error']}{印出}"


@tool
def run_code(code: str) -> str:
    """安全地執行一段 Python 作業，回傳輸出或錯誤訊息。

    什麼時候用：想知道作業「實際跑起來」會印出什麼、會不會出錯時。
    參數 code：完整的 Python 程式碼（字串）。
    回傳：一行結果，例如「✅ 執行成功，輸出：5」或「❌ 第 3 行 NameError: ...」。
    限制：不能 import os、sys、subprocess 等模組；超過 3 秒會被停止。
    """
    return 執行作業(code)
''',
    'tools/check_indent.py': r'''"""check_indent：檢查縮排與每行長度。

故意「只」檢查縮排和行長、不碰變數命名：命名檢查留給第 2 堂 2-B，
用來比較「寫在手冊文字裡」和「寫成腳本」的差別。
"""

from langchain.tools import tool

行長上限 = 79


def 檢查縮排(code: str, 上限: int = 行長上限) -> str:
    問題 = []
    for i, line in enumerate(code.splitlines(), start=1):
        if not line.strip():
            continue
        開頭空白 = line[: len(line) - len(line.lstrip())]
        if "\t" in 開頭空白:
            問題.append(f"第 {i} 行：用了 Tab 縮排，建議改成 4 個空白")
        elif len(開頭空白) % 4 != 0:
            問題.append(f"第 {i} 行：縮排 {len(開頭空白)} 格，建議用 4 的倍數")
        if len(line) > 上限:
            問題.append(f"第 {i} 行：有 {len(line)} 個字元，超過 {上限}")
    return "\n".join(問題) if 問題 else "OK：縮排與行長都沒問題"


@tool
def check_indent(code: str) -> str:
    """檢查 Python 作業的縮排（是否為 4 的倍數、有沒有混用 Tab）與每行長度（上限 79 字元）。

    什麼時候用：想確認作業的排版整不整齊時。不會檢查變數命名，也不會執行程式。
    參數 code：完整的 Python 程式碼（字串）。
    回傳：每個問題一行，例如「第 2 行：縮排 2 格，建議用 4 的倍數」；全部沒問題時回傳「OK：…」。
    """
    return 檢查縮排(code)
''',
    'tools/read_spec.py': r'''"""read_spec：讀這一題的作業要求。

題目要求存在 review-bot/spec.md，notebook 的表單改了題目就會重寫這個檔案。
"""

from pathlib import Path

from langchain.tools import tool

題目檔 = Path(__file__).resolve().parent.parent / "spec.md"


def 讀題目() -> str:
    if not 題目檔.exists():
        return "（這一題沒有提供作業要求）"
    return 題目檔.read_text(encoding="utf-8").strip() or "（這一題沒有提供作業要求）"


@tool
def read_spec() -> str:
    """讀取這一題的作業要求（題目寫了什麼、要交什麼）。

    什麼時候用：要判斷作業有沒有做到題目要求之前，例如函式名稱、輸出格式對不對。
    不需要參數。
    回傳：題目要求的全文。
    """
    return 讀題目()
''',
    'agent.py': r'''"""1-B：作業小助教的 agent 版，路線由 AI 決定。

Agent = 模型 + 工具 + 迴圈。
- 建立_agent()      ：LangChain 1.0 的標準寫法 create_agent，一行做出 agent
- 建立_展開版_agent()：同一個 agent 自己用 LangGraph 組出來，看清楚底下的迴圈
- pretty_trace()    ：把每一步攤開印出來，看它選了哪個工具
"""

from langchain.agents import create_agent
from langchain_core.messages import SystemMessage
from langgraph.graph import START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition

from tools.check_indent import check_indent
from tools.read_spec import read_spec
from tools.run_code import run_code

基本_system_prompt = (
    "你是 Python 作業小助教。學生是剛學會 Python 的初學者，請用繁體中文、溫和的語氣給回饋。"
    "你有幾個工具可以用：要不要用、用哪一個、用幾次，由你自己決定。"
    "回饋分兩段：第一段以「優點：」開頭，第二段以「改進：」開頭。"
)
小助教的工具 = [run_code, check_indent, read_spec]


def 建立_agent(model, system_prompt: str = 基本_system_prompt, tools=None):
    return create_agent(model, tools=tools or 小助教的工具, system_prompt=system_prompt)


def 建立_展開版_agent(model, system_prompt: str = 基本_system_prompt, tools=None):
    """教學用的核心迴圈：展開模型與工具如何互動，不是 create_agent 的完整實作。"""
    tools = tools or 小助教的工具
    model_with_tools = model.bind_tools(tools)

    def ask_model(state: MessagesState) -> dict:
        messages = [SystemMessage(system_prompt)] + state["messages"]
        return {"messages": [model_with_tools.invoke(messages)]}

    builder = StateGraph(MessagesState)
    builder.add_node("ask_model", ask_model)
    builder.add_node("tools", ToolNode(tools))
    builder.add_edge(START, "ask_model")
    # tools_condition：上一則訊息有 tool_calls 就走 tools，否則走 END
    builder.add_conditional_edges("ask_model", tools_condition)
    builder.add_edge("tools", "ask_model")  # tools 跑完繞回 ask_model
    return builder.compile()


def 作業訊息(作業: str) -> str:
    return f"幫我看這段作業：\n```python\n{作業}\n```"


def pretty_trace(graph, user_message: str, config: dict | None = None) -> list[dict]:
    events = []
    print(f"USER: {user_message}\n")
    inputs = {"messages": [{"role": "user", "content": user_message}]}
    stream = graph.stream(inputs, config=config or {}, stream_mode="updates")
    for step, event in enumerate(stream, start=1):
        events.append(event)
        for node_name, node_state in event.items():
            print(f"--- 第 {step} 步 | 節點: {node_name} ---")
            for msg in (node_state or {}).get("messages", []):
                role = type(msg).__name__
                tool_calls = getattr(msg, "tool_calls", None)
                if tool_calls:
                    for call in tool_calls:
                        print(f"  模型決定呼叫 -> {call['name']}({call['args']})")
                else:
                    text = msg.text if hasattr(msg, "text") else str(msg.content)
                    if len(text) > 400:
                        text = text[:400] + "..."
                    print(f"  [{role}] {text}")
            print()
    return events


def 工具順序(messages_or_events) -> list[str]:
    """從 messages（或 pretty_trace 的 events）抽出工具呼叫的順序。"""
    messages = []
    for item in messages_or_events:
        if isinstance(item, dict):
            for node_state in item.values():
                messages += (node_state or {}).get("messages", [])
        else:
            messages.append(item)
    return [c["name"] for m in messages for c in (getattr(m, "tool_calls", None) or [])]


def 最後回覆(messages_or_events) -> str:
    messages = []
    for item in messages_or_events:
        if isinstance(item, dict):
            for node_state in item.values():
                messages += (node_state or {}).get("messages", [])
        else:
            messages.append(item)
    for m in reversed(messages):
        if m.type == "ai" and not getattr(m, "tool_calls", None):
            return m.text
    return ""
''',
    'spec.md': r'''請寫一個函式 add(a, b)，回傳兩個數字相加的結果，並用 print(add(2, 3)) 印出 5。
''',
}
for rel, content in _檔案.items():
    (RB / rel).write_text(content, encoding="utf-8")
sys.path.insert(0, str(RB))
# 剛寫好的檔案要讓 Python 重新找一次（如果之前先跑過別的格子，Python 會記得「這個資料夾不存在」）
import importlib
importlib.invalidate_caches()

# 3. 記住模型來源（不會存 API key），測試連線
import llm
llm.存設定(模型來源, colab_模型名稱.strip(), gemini_模型名稱.strip())
if 講師_API_KEY.strip():
    os.environ["GOOGLE_API_KEY"] = 講師_API_KEY.strip()
from workshop import *
model = 準備()

回覆 = model.invoke("用一句話跟剛學 Python 的同學打招呼").text
print(f"✅ 環境設定完成，模型來源：{模型來源}")
print(f"🤖 模型說：{回覆.strip()}")

### ✏️（選用）放進你自己的作業

課後作業要把**你自己寫過的一段作業**丟進來跑。把下面那格的程式換成你的作業、執行它，
之後在表單的「作業」選 **我的作業** 就會用它。課堂上可以先跳過這格。

In [2]:
%%writefile /content/review-bot/我的作業.py
# 把這裡換成你自己寫過的一段 Python 作業，改完記得執行這一格
def average(scores):
    total = 0
    for s in scores:
        total = total + s
    return total / len(scores)

print(average([90, 85, 77]))


Writing /content/review-bot/我的作業.py


---
## 第 1 段：工具箱，以及 run_code 的圍欄

> 📍 **對應簡報**：投影片 31「換個做法：不畫流程圖，給它一個工具箱」、投影片 32「先試工具，再交給 agent」

先準備 agent 的工具箱。

下面三格會寫出三把工具（不用改，直接執行）。每把工具都是一個普通的 Python 函式，
上面加一行 `@tool`，再把它加入 agent 的工具清單，模型便能要求使用它。

`run_code` 是**真的會執行程式**的工具，而學生作業裡什麼都可能有，所以先看兩個執行限制，以及一個方便閱讀的設計：
1. 在執行前檢查允許的語法與模組，擋下這堂課測試的危險操作
2. 設時間上限（timeout）：超過 3 秒就強制停止，無限迴圈也停得下來
3. 把錯誤訊息縮成一行，方便人與模型讀取；這不是安全限制

### 🤔 預測題
下面會丟四段作業給 `run_code`：正常的、打錯變數的、偷偷 `import os` 的、無限迴圈的。各會得到什麼結果？

In [3]:
%%writefile /content/review-bot/tools/run_code.py
"""run_code：在籠子裡執行學生的作業。

學生作業裡什麼都可能有（import os 刪檔、無限迴圈），所以這個工具有三道圍欄：
1. 執行前先檢查：不准 import os / sys / subprocess / shutil 這類危險模組，
   也不准用 eval、exec、open 和底線開頭的屬性（常被拿來繞過限制）
2. 開一個獨立的子程序執行，超過時間就強制停止（無限迴圈也停得下來）
3. 只回一行結果（第幾行、什麼錯），模型才看得懂

⚠️ 這是教學用的圍欄。正式產品請用真正的 sandbox（容器、RestrictedPython、e2b、Modal）。
"""

import ast
import json
import os
import subprocess
import sys
import tempfile

from langchain.tools import tool

禁止的模組 = {"os", "sys", "subprocess", "shutil", "socket", "pathlib", "ctypes", "importlib",
           "multiprocessing", "threading", "signal", "pickle", "builtins", "io", "inspect", "gc"}
允許的模組 = {"math", "random", "string", "re", "datetime", "collections", "itertools", "functools",
           "statistics", "time", "decimal", "fractions", "json", "copy", "heapq", "bisect"}
禁止的函式 = {"eval", "exec", "compile", "open", "input", "__import__", "globals", "locals", "vars",
           "getattr", "setattr", "delattr", "breakpoint", "help", "exit", "quit", "memoryview"}
禁止的屬性 = {"system", "popen", "modules", "builtins", "f_globals", "f_back", "gi_frame", "cr_frame"}
最長時間_秒 = 3
最長輸出_字 = 800

# 子程序裡真正執行作業的那段程式：用「縮小版的 builtins」執行，import 只放行白名單
_RUNNER = r"""
import builtins, contextlib, io, json, sys, traceback
code = sys.stdin.buffer.read().decode("utf-8")
allowed = set(json.loads(sys.argv[1]))
blocked = set(json.loads(sys.argv[2]))
real_import = builtins.__import__
def guarded_import(name, globals=None, locals=None, fromlist=(), level=0):
    if name.split(".")[0] not in allowed:
        raise ImportError("這個工具不允許 import " + name)
    return real_import(name, globals, locals, fromlist, level)
safe = {k: v for k, v in vars(builtins).items() if k not in blocked}
safe["__import__"] = guarded_import
try:
    import resource
    resource.setrlimit(resource.RLIMIT_AS, (512 * 1024 * 1024, 512 * 1024 * 1024))
except Exception:
    pass
out = io.StringIO()
result = {"ok": True}
try:
    compiled = compile(code, "作業.py", "exec")
    with contextlib.redirect_stdout(out):
        exec(compiled, {"__builtins__": safe, "__name__": "__main__"})
except SyntaxError as e:
    result = {"ok": False, "line": e.lineno, "error": "SyntaxError: " + str(e.msg)}
except BaseException as e:
    line = None
    for frame in traceback.extract_tb(e.__traceback__):
        if frame.filename == "作業.py":
            line = frame.lineno
    result = {"ok": False, "line": line, "error": type(e).__name__ + ": " + str(e)}
result["stdout"] = out.getvalue()
sys.__stdout__.write(json.dumps(result))
"""


def _先檢查(code: str) -> str | None:
    """執行前的靜態檢查；有問題就回一行說明，沒問題回 None。"""
    try:
        tree = ast.parse(code)
    except SyntaxError as e:
        return f"❌ 第 {e.lineno} 行 SyntaxError: {e.msg}"
    for node in ast.walk(tree):
        if isinstance(node, (ast.Import, ast.ImportFrom)):
            names = [a.name for a in node.names] if isinstance(node, ast.Import) else [node.module or ""]
            for name in names:
                root = name.split(".")[0]
                if root in 禁止的模組 or root not in 允許的模組:
                    return f"⛔ 已擋下：第 {node.lineno} 行 import {name}（這個工具不允許使用 {root}）"
        elif isinstance(node, ast.Name) and node.id in 禁止的函式:
            return f"⛔ 已擋下：第 {node.lineno} 行用了 {node.id}（這個工具不允許）"
        elif isinstance(node, ast.Attribute) and (
            node.attr.startswith("_") or node.attr in 禁止的屬性 or node.attr in 禁止的模組
        ):
            return f"⛔ 已擋下：第 {node.lineno} 行存取了 .{node.attr}（這個工具不允許）"
    return None


def 執行作業(code: str, 最長時間: float = 最長時間_秒) -> str:
    """真正做事的函式：檢查 → 子程序執行 → 回傳一行結果。"""
    if not code.strip():
        return "❌ 作業是空白的，沒有東西可以執行"
    擋下 = _先檢查(code)
    if 擋下:
        return 擋下
    env = {"PYTHONIOENCODING": "utf-8", "PATH": os.environ.get("PATH", "")}
    if os.name == "nt":
        env["SYSTEMROOT"] = os.environ.get("SYSTEMROOT", "")
    try:
        proc = subprocess.run(
            [sys.executable, "-I", "-c", _RUNNER, json.dumps(sorted(允許的模組)), json.dumps(sorted(禁止的函式))],
            input=code, capture_output=True, text=True, encoding="utf-8",
            timeout=最長時間, cwd=tempfile.gettempdir(), env=env,
        )
    except subprocess.TimeoutExpired:
        return f"⏱️ 執行超過 {最長時間:g} 秒被強制停止（可能有無限迴圈）"
    try:
        result = json.loads(proc.stdout.strip().splitlines()[-1])
    except (IndexError, json.JSONDecodeError):
        return "❌ 執行失敗（程式沒有正常結束，可能用了太多記憶體）"
    stdout = result.get("stdout", "")
    if len(stdout) > 最長輸出_字:
        stdout = stdout[:最長輸出_字] + "\n…（輸出太長，後面省略）"
    if result["ok"]:
        return f"✅ 執行成功，輸出：\n{stdout}" if stdout else "✅ 執行成功（沒有印出任何東西）"
    where = f"第 {result['line']} 行 " if result.get("line") else ""
    印出 = f"\n（出錯前印出：{stdout.strip()}）" if stdout.strip() else ""
    return f"❌ {where}{result['error']}{印出}"


@tool
def run_code(code: str) -> str:
    """安全地執行一段 Python 作業，回傳輸出或錯誤訊息。

    什麼時候用：想知道作業「實際跑起來」會印出什麼、會不會出錯時。
    參數 code：完整的 Python 程式碼（字串）。
    回傳：一行結果，例如「✅ 執行成功，輸出：5」或「❌ 第 3 行 NameError: ...」。
    限制：不能 import os、sys、subprocess 等模組；超過 3 秒會被停止。
    """
    return 執行作業(code)


Overwriting /content/review-bot/tools/run_code.py


In [4]:
%%writefile /content/review-bot/tools/check_indent.py
"""check_indent：檢查縮排與每行長度。

故意「只」檢查縮排和行長、不碰變數命名：命名檢查留給第 2 堂 2-B，
用來比較「寫在手冊文字裡」和「寫成腳本」的差別。
"""

from langchain.tools import tool

行長上限 = 79


def 檢查縮排(code: str, 上限: int = 行長上限) -> str:
    問題 = []
    for i, line in enumerate(code.splitlines(), start=1):
        if not line.strip():
            continue
        開頭空白 = line[: len(line) - len(line.lstrip())]
        if "\t" in 開頭空白:
            問題.append(f"第 {i} 行：用了 Tab 縮排，建議改成 4 個空白")
        elif len(開頭空白) % 4 != 0:
            問題.append(f"第 {i} 行：縮排 {len(開頭空白)} 格，建議用 4 的倍數")
        if len(line) > 上限:
            問題.append(f"第 {i} 行：有 {len(line)} 個字元，超過 {上限}")
    return "\n".join(問題) if 問題 else "OK：縮排與行長都沒問題"


@tool
def check_indent(code: str) -> str:
    """檢查 Python 作業的縮排（是否為 4 的倍數、有沒有混用 Tab）與每行長度（上限 79 字元）。

    什麼時候用：想確認作業的排版整不整齊時。不會檢查變數命名，也不會執行程式。
    參數 code：完整的 Python 程式碼（字串）。
    回傳：每個問題一行，例如「第 2 行：縮排 2 格，建議用 4 的倍數」；全部沒問題時回傳「OK：…」。
    """
    return 檢查縮排(code)


Overwriting /content/review-bot/tools/check_indent.py


In [5]:
%%writefile /content/review-bot/tools/read_spec.py
"""read_spec：讀這一題的作業要求。

題目要求存在 review-bot/spec.md，notebook 的表單改了題目就會重寫這個檔案。
"""

from pathlib import Path

from langchain.tools import tool

題目檔 = Path(__file__).resolve().parent.parent / "spec.md"


def 讀題目() -> str:
    if not 題目檔.exists():
        return "（這一題沒有提供作業要求）"
    return 題目檔.read_text(encoding="utf-8").strip() or "（這一題沒有提供作業要求）"


@tool
def read_spec() -> str:
    """讀取這一題的作業要求（題目寫了什麼、要交什麼）。

    什麼時候用：要判斷作業有沒有做到題目要求之前，例如函式名稱、輸出格式對不對。
    不需要參數。
    回傳：題目要求的全文。
    """
    return 讀題目()


Overwriting /content/review-bot/tools/read_spec.py


In [ ]:
#@title ▶ 執行：試試 run_code 的限制與錯誤訊息
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # 準備、記下…（「0. 環境設定」寫好的小工具）
except ImportError:
    raise SystemExit("⚠️ 找不到環境：請先執行最上面的「0. 環境設定」那一格")
model = 準備()
from tools.run_code import run_code

四段作業 = {
    "正常的作業": "print(sum([1, 2, 3]))",
    "打錯變數": "total = 1 + 2\nprint(totla)",
    "偷偷 import os": "import os\nos.remove('重要檔案.txt')",
    "無限迴圈": "while True:\n    pass",
}
結果們 = {}
for 名稱, 作業 in 四段作業.items():
    結果們[名稱] = run_code.invoke({"code": 作業})
    print(f"【{名稱}】\n{作業}\n→ {結果們[名稱]}\n")
記下("1-B 第 1 段：run_code 的圍欄", 結果們)

### 👀 你看到了什麼

- 正常的作業印出 `6`；打錯變數只回一行「第 2 行 NameError…」，不是一大段 traceback。
- `import os` 在**執行前**就被擋下，檔案一個都沒動到；無限迴圈 3 秒後被強制停止。
- 執行外部程式碼需要隔離與權限限制：模型可能受輸入影響，要求 `run_code` 執行不適當的操作（第 2 堂 2-C 會回來講這件事）。
  這裡的圍欄是教學用的簡化版，正式產品要用真正的 sandbox（容器、e2b、Modal 等）。

---
## 第 2 段：模型會拿到哪些工具資訊？

> 📍 **對應簡報**：投影片 33「工具介面交給模型的是這幾項資訊」

上一段寫好了三把工具。工具介面不會自動傳送函式實作；模型如何知道每把工具能做什麼？

`@tool` 會把函式名稱、參數、還有函式開頭三引號裡的那段說明（叫 docstring）包成一份**說明書**交給模型。
模型會參考這份說明書、任務與對話，決定是否要求使用工具。

### 🤔 預測題
模型看得到 `run_code` 裡面「禁止 import os」那段程式嗎？它怎麼知道 `read_spec` 是拿來讀題目的？

In [7]:
#@title ▶ 執行：印出三個工具的說明書
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # 準備、記下…（「0. 環境設定」寫好的小工具）
except ImportError:
    raise SystemExit("⚠️ 找不到環境：請先執行最上面的「0. 環境設定」那一格")
model = 準備()
from tools.check_indent import check_indent
from tools.read_spec import read_spec
from tools.run_code import run_code

for t in [run_code, check_indent, read_spec]:
    print("name：", t.name)
    print("description：", t.description)
    print("args：", t.args)
    print("-" * 60)

name： run_code
description： 安全地執行一段 Python 作業，回傳輸出或錯誤訊息。

什麼時候用：想知道作業「實際跑起來」會印出什麼、會不會出錯時。
參數 code：完整的 Python 程式碼（字串）。
回傳：一行結果，例如「✅ 執行成功，輸出：5」或「❌ 第 3 行 NameError: ...」。
限制：不能 import os、sys、subprocess 等模組；超過 3 秒會被停止。
args： {'code': {'title': 'Code', 'type': 'string'}}
------------------------------------------------------------
name： check_indent
description： 檢查 Python 作業的縮排（是否為 4 的倍數、有沒有混用 Tab）與每行長度（上限 79 字元）。

什麼時候用：想確認作業的排版整不整齊時。不會檢查變數命名，也不會執行程式。
參數 code：完整的 Python 程式碼（字串）。
回傳：每個問題一行，例如「第 2 行：縮排 2 格，建議用 4 的倍數」；全部沒問題時回傳「OK：…」。
args： {'code': {'title': 'Code', 'type': 'string'}}
------------------------------------------------------------
name： read_spec
description： 讀取這一題的作業要求（題目寫了什麼、要交什麼）。

什麼時候用：要判斷作業有沒有做到題目要求之前，例如函式名稱、輸出格式對不對。
不需要參數。
回傳：題目要求的全文。
args： {}
------------------------------------------------------------


### 👀 你看到了什麼

- 這裡的工具介面提供 `name`、`description`（docstring）和 `args`，不會自動把函式實作交給模型。
- 好的 docstring 要寫四件事：**做什麼、什麼時候該呼叫、參數是什麼意思、回傳什麼**。
  只寫一句「執行程式」、沒寫什麼時候用，模型可能漏用或誤用它。
- 第 2 堂的 Skill 也有名稱與說明，協助模型判斷是否讀取完整指引。

---
## 第 3 段：一行做出 agent，看它每一步選了什麼

> 📍 **對應簡報**：投影片 34「一行做出 agent：create_agent」、投影片 35「讀懂 agent 的執行紀錄」

前兩段準備好了工具箱與說明書。這一段把模型和工具箱組起來，正式做出 agent，看它批改一份作業時每一步選了什麼。

下一格寫出 `agent.py`：裡面的 `建立_agent` 只有一行 `create_agent(model, tools=..., system_prompt=...)`，
這堂課使用這個介面：

- `tools`：工具箱（三把本地工具）
- `system_prompt`：放在對話最前面、每次都會帶著的那段指示，例如「你是 Python 作業小助教，語氣要溫和」
- `pretty_trace`：把 agent 的每一步攤開印出來，讓你看到模型提出的工具要求與工具結果；這不是模型內部的思考過程

### 🤔 預測題（寫在學習單上）
作業選「變數打錯字」（`return summ`，但變數叫 `total`），題目要求是「寫一個 add(a, b)」。
1. 它**第一個**會用哪個工具？
2. 它總共會用幾次工具？三把工具都會用到嗎？

In [ ]:
%%writefile /content/review-bot/agent.py
"""1-B：作業小助教的 agent 版，路線由 AI 決定。

Agent = 模型 + 工具 + 迴圈。
- 建立_agent()      ：LangChain 1.0 的標準寫法 create_agent，一行做出 agent
- 建立_展開版_agent()：同一個 agent 自己用 LangGraph 組出來，看清楚底下的迴圈
- pretty_trace()    ：把每一步攤開印出來，看它選了哪個工具
"""

from langchain.agents import create_agent
from langchain_core.messages import SystemMessage
from langgraph.graph import START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition

from tools.check_indent import check_indent
from tools.read_spec import read_spec
from tools.run_code import run_code

基本_system_prompt = (
    "你是 Python 作業小助教。學生是剛學會 Python 的初學者，請用繁體中文、溫和的語氣給回饋。"
    "你有幾個工具可以用：要不要用、用哪一個、用幾次，由你自己決定。"
    "回饋分兩段：第一段以「優點：」開頭，第二段以「改進：」開頭。"
)
小助教的工具 = [run_code, check_indent, read_spec]


def 建立_agent(model, system_prompt: str = 基本_system_prompt, tools=None):
    return create_agent(model, tools=tools or 小助教的工具, system_prompt=system_prompt)


def 建立_展開版_agent(model, system_prompt: str = 基本_system_prompt, tools=None):
    """教學用的核心迴圈：展開模型與工具如何互動，不是 create_agent 的完整實作。"""
    tools = tools or 小助教的工具
    model_with_tools = model.bind_tools(tools)

    def ask_model(state: MessagesState) -> dict:
        messages = [SystemMessage(system_prompt)] + state["messages"]
        return {"messages": [model_with_tools.invoke(messages)]}

    builder = StateGraph(MessagesState)
    builder.add_node("ask_model", ask_model)
    builder.add_node("tools", ToolNode(tools))
    builder.add_edge(START, "ask_model")
    # tools_condition：上一則訊息有 tool_calls 就走 tools，否則走 END
    builder.add_conditional_edges("ask_model", tools_condition)
    builder.add_edge("tools", "ask_model")  # tools 跑完繞回 ask_model
    return builder.compile()


def 作業訊息(作業: str) -> str:
    return f"幫我看這段作業：\n```python\n{作業}\n```"


def pretty_trace(graph, user_message: str, config: dict | None = None) -> list[dict]:
    events = []
    print(f"USER: {user_message}\n")
    inputs = {"messages": [{"role": "user", "content": user_message}]}
    stream = graph.stream(inputs, config=config or {}, stream_mode="updates")
    for step, event in enumerate(stream, start=1):
        events.append(event)
        for node_name, node_state in event.items():
            print(f"--- 第 {step} 步 | 節點: {node_name} ---")
            for msg in (node_state or {}).get("messages", []):
                role = type(msg).__name__
                tool_calls = getattr(msg, "tool_calls", None)
                if tool_calls:
                    for call in tool_calls:
                        print(f"  模型決定呼叫 -> {call['name']}({call['args']})")
                else:
                    text = msg.text if hasattr(msg, "text") else str(msg.content)
                    if len(text) > 400:
                        text = text[:400] + "..."
                    print(f"  [{role}] {text}")
            print()
    return events


def 工具順序(messages_or_events) -> list[str]:
    """從 messages（或 pretty_trace 的 events）抽出工具呼叫的順序。"""
    messages = []
    for item in messages_or_events:
        if isinstance(item, dict):
            for node_state in item.values():
                messages += (node_state or {}).get("messages", [])
        else:
            messages.append(item)
    return [c["name"] for m in messages for c in (getattr(m, "tool_calls", None) or [])]


def 最後回覆(messages_or_events) -> str:
    messages = []
    for item in messages_or_events:
        if isinstance(item, dict):
            for node_state in item.values():
                messages += (node_state or {}).get("messages", [])
        else:
            messages.append(item)
    for m in reversed(messages):
        if m.type == "ai" and not getattr(m, "tool_calls", None):
            return m.text
    return ""


In [ ]:
#@title ▶ 執行：create_agent 做出的 agent，看它每一步
作業 = "變數打錯字"  #@param ["變數打錯字", "正確的程式", "縮排亂掉", "少了右括號", "我的作業"]
題目要求 = "請寫一個函式 add(a, b)，回傳兩個數字相加的結果，並用 print(add(2, 3)) 印出 5。"  #@param {type:"string"}
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # 準備、記下…（「0. 環境設定」寫好的小工具）
except ImportError:
    raise SystemExit("⚠️ 找不到環境：請先執行最上面的「0. 環境設定」那一格")
model = 準備()
import agent

(BASE / "spec.md").write_text(題目要求, encoding="utf-8")   # read_spec 會讀這個檔案
小助教 = agent.建立_agent(model)
events = agent.pretty_trace(小助教, agent.作業訊息(取得作業(作業)))
print("這次用工具的順序：", " → ".join(agent.工具順序(events)) or "（沒有用任何工具）")
記下(f"1-B 第 3 段：create_agent（{作業}）", events)

### 👀 你看到了什麼

- 每一個「節點: model」是一次模型回應；「節點: tools」是你的程式在執行它選的工具。一來一回就是 1-0 那個迴圈轉一圈。
- 最後一步的 `AIMessage` 沒有再叫工具，迴圈就結束了。
- 對照你的預測：它先用了哪一個？有沒有哪把工具根本沒用？下一段會看到這個順序每次可能不一樣。
- `invoke` 只給你終點；`stream`（`pretty_trace` 用的）把過程攤開。

---
## 第 4 段：同一段作業跑三次，順序會一樣嗎？

> 📍 **對應簡報**：投影片 36「同一段作業跑三次，順序會一樣嗎？」

上一段看了 agent 走一次的路線。同一段作業再跑三次，路線會一樣嗎？

這是 agent 和 workflow 最大的差別：**路線不是你畫的，是它當下決定的**。
這一段也可以改 **system prompt**（每次都放在最前面的那段指示）加一句話，觀察新增指示如何影響工具順序。

### 🤔 預測題（兩輪）
1. 先讓「system prompt 加一句」保持空白，執行。三次用工具的順序會一樣嗎？
2. 再把它改成 **一定要先讀題目要求**，執行。這次順序有沒有變？三次都一樣了嗎？

In [ ]:
#@title ▶ 執行：同一段作業跑三次，比較用工具的順序
作業 = "變數打錯字"  #@param ["變數打錯字", "正確的程式", "縮排亂掉", "少了右括號", "我的作業"]
system_prompt_加一句 = ""  #@param ["", "一定要先讀題目要求", "一定要先執行作業看結果", "不要使用任何工具"] {allow-input: true}
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # 準備、記下…（「0. 環境設定」寫好的小工具）
except ImportError:
    raise SystemExit("⚠️ 找不到環境：請先執行最上面的「0. 環境設定」那一格")
model = 準備()
import agent

system_prompt = agent.基本_system_prompt + system_prompt_加一句
print("system prompt：", system_prompt, "\n")
小助教 = agent.建立_agent(model, system_prompt=system_prompt)

三次 = []
for i in range(3):
    result = 小助教.invoke({"messages": [{"role": "user", "content": agent.作業訊息(取得作業(作業))}]})
    順序 = agent.工具順序(result["messages"])
    三次.append(順序)
    print(f"========== 第 {i + 1} 次 ==========")
    print("用工具的順序：", " → ".join(順序) or "（沒有用任何工具，直接評語）")
    print(result["messages"][-1].text[:300])
    print()

print("三次順序都一樣嗎？", "是" if all(x == 三次[0] for x in 三次) else "不一樣")
記下(f"1-B 第 4 段：跑三次（加一句：{system_prompt_加一句 or '無'}）", {"三次的工具順序": 三次})

### 👀 你看到了什麼

- 三次可能相同，也可能不同；先記錄實際工具順序與漏掉的檢查。結果相同不代表之後都一樣；若漏掉必要步驟，仍是系統需要處理的問題。
- 加了「一定要先讀題目要求」之後，觀察第一步是否改為 `read_spec`；**不能把指示當作執行保證**：
  prompt 可以影響選擇；必須執行的檢查，應由流程安排，並驗證它的結果。
- 所以接下來要想清楚：哪些步驟由流程控制、哪些交給 AI 判斷（簡報的第四站）。

---
## 第 5 段：它會繞圈：recursion limit

> 📍 **對應簡報**：投影片 37「如果工具一直失敗，agent 會停嗎？」、38「設定上限，再判斷為什麼停止」

上一段看到 agent 的路線由模型決定，可能改變。最糟的情況是：它一直繞圈，自己停不下來。

下面這格給 agent 一個**故意永遠失敗**的工具 `always_fail`，再叫它「一定要成功才停」。
我們在程式裡設 `recursion_limit=25` 來限制圖的執行步數；這不是「最多呼叫 25 次工具」，模型節點也會占步數。模型也可能先自行停止。

### 🤔 預測題
模型會自行停止，還是持續呼叫工具直到達上限？步數與工具呼叫次數會相同嗎？

In [ ]:
#@title ▶ 執行：一個永遠失敗的工具，觀察自行停止或達到 recursion limit
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # 準備、記下…（「0. 環境設定」寫好的小工具）
except ImportError:
    raise SystemExit("⚠️ 找不到環境：請先執行最上面的「0. 環境設定」那一格")
model = 準備()
from langchain.agents import create_agent
from langchain.tools import tool
from langgraph.errors import GraphRecursionError

@tool
def always_fail() -> str:
    """做一件很重要的事。失敗的話請再試一次，直到成功為止。"""
    return "執行錯誤：RuntimeError: 還是不行，請再試一次"

繞圈agent = create_agent(
    model, tools=[always_fail],
    system_prompt="你的任務是呼叫 always_fail 直到它成功為止。不管失敗幾次都要再試，不要放棄，也不要用文字回答。",
)
次數 = 0
try:
    for step, event in enumerate(繞圈agent.stream({"messages": [{"role": "user", "content": "開始吧"}]},
                                               config={"recursion_limit": 25}, stream_mode="updates"), start=1):
        for node, update in event.items():
            for msg in (update or {}).get("messages", []):
                if getattr(msg, "tool_calls", None):
                    次數 += len(msg.tool_calls)
                    print(f"--- 第 {step} 步 | 節點: {node} ---  模型決定呼叫 -> always_fail()")
                elif msg.type == "tool":
                    print(f"--- 第 {step} 步 | 節點: {node} ---  [ToolMessage] {msg.text}")
                else:
                    print(f"--- 第 {step} 步 | 節點: {node} ---  [AIMessage] {msg.text[:80]}")
    print("\n（這次模型自行停止，沒有達到上限；工具仍未成功。請記錄這個結果。）")
except GraphRecursionError as e:
    print(f"\n🛑 GraphRecursionError：{str(e).splitlines()[0]}")
print(f"模型總共呼叫了 always_fail {次數} 次")
記下("1-B 第 5 段：recursion limit", {"呼叫次數": 次數})

### 👀 你看到了什麼

- 若模型持續叫 `always_fail`，每一圈都會增加模型呼叫成本；也可能自行停止。
- 持續執行而達到步數上限時，會得到 `GraphRecursionError`。停止不等於任務成功。
- 1-0 的「最多 5 圈」、1-A 的「最多生成次數」、這裡的 `recursion_limit`，都是在程式中設停止上限；三者計算的單位不同。
- 接著看 MCP 示範，再到簡報第四站討論：哪些步驟由流程控制、哪些交給 AI 判斷。

---
## 第 6 段（教師示範）：Tool 從哪裡來：自己寫的函式 vs MCP 插座

> 📍 **對應簡報**：投影片 39「Tool 從哪裡來：自己寫的函式 vs MCP 插座」

前面先用本地工具建立 agent，並觀察執行過程與停止條件。現在擴充工具的來源。**說明書不一定是你寫的**：別人做好的工具，也能從插座接進來。

- **本地函式**：你自己寫、住在同一個程式裡（我們的三把工具）。
- **MCP**：工具住在**另一個程式或服務**裡（例如 GitHub），透過一個大家共用的「插座」規格接進來。
  **MCP 不是另一種 agent，它是「工具」的外接方式。**
- 代價：交給模型的工具越多，工具說明書通常也越長。連上 MCP 不代表必須把所有工具都交給模型。
  （桌子＝context window，模型一次能處理的 token 上限；說明書越多，留給作業和回饋的空間越少。）

這一段真的接一個 MCP server。下一格寫出 `deploy/homework_mcp.py`：一個假的「作業倉庫」，
有三個工具：`list_homework`（誰交了作業）、`get_homework`（拿某位同學的作業）、`send_email`（寄信，假的，只寫進 `寄件匣.txt`）。
打開看就會發現：**它就是幾個普通的 Python 函式，上面加了 `@mcp.tool()`**，然後在自己的程式裡跑起來。

接著用 `langchain-mcp-adapters` 這個套件把插座接上，拿到的工具和三把本地工具一起交給 agent。
（`create_agent` 是「把模型和工具箱組成 agent」的函式，前面的本地 agent 已經用過它。
程式裡的 `await` 意思是「等這件事做完再往下」：MCP 工具要透過插座和另一個程式溝通，所以要用 `await`，Colab 的格子可以直接寫。）

### 🤔 預測題
1. 從插座接進來的工具，agent 用起來和自己寫的 `@tool` 有什麼不一樣？
2. 請 agent「看小美交的作業、跑跑看有沒有錯」，它會先叫哪一個工具？會不會同時用到 MCP 的工具和本地的 `run_code`？

In [8]:
%%writefile /content/review-bot/deploy/homework_mcp.py
"""作業倉庫 MCP server：工具住在「另一個程式」裡，透過 MCP 插座接進來。

它就是一個普通的 Python 檔，只是函式上面加了 @mcp.tool()。
真實世界裡，這個角色是 GitHub 的 MCP server（抓作業）和信箱的 MCP server（寄信）；
這裡用假資料代替，寄信也只會寫進 寄件匣.txt。

兩種接法：
    python homework_mcp.py              # stdio：本機插座，由 agent 那邊把它開起來
    python homework_mcp.py --http 8765  # Streamable HTTP：變成一個網址，像遠端 MCP
"""

import argparse
import datetime as _dt
import json
from pathlib import Path

from mcp.server.fastmcp import FastMCP

寄件匣 = Path(__file__).resolve().parent.parent / "寄件匣.txt"

# 假的作業倉庫：學生、信箱、交的作業
作業們 = {
    "小明": ("xiaoming@example.com", "print('hi'\n"),
    "小華": ("xiaohua@example.com", "def add(a, b):\n    return a + b\n\nprint(add(2, 3))\n"),
    "小美": ("xiaomei@example.com", "def add(a, b):\n    total = a + b\n    return summ\n\nprint(add(2, 3))\n"),
    "阿傑": ("ajie@example.com", "def CalcAverage(scoreList):\n    Total = 0\n    for s in scoreList:\n        Total += s\n"
                                 "    return Total / len(scoreList)\n\nmyScores = [90, 85, 77]\nprint(CalcAverage(myScores))\n"),
    "小芳": ("xiaofang@example.com", "def average(scores):\n  total = 0\n  for s in scores:\n        total += s\n"
                                     "  return total / len(scores)\n\nprint(average([90, 85, 77]))\n"),
}

mcp = FastMCP("作業倉庫", log_level="WARNING")


@mcp.tool()
def list_homework() -> str:
    """列出已經交作業的學生（名字和 email），回傳 JSON 字串。要批改作業時先呼叫這個。"""
    return json.dumps([{"學生": name, "email": email} for name, (email, _) in 作業們.items()], ensure_ascii=False)


@mcp.tool()
def get_homework(student: str) -> str:
    """取得某位學生交的作業程式碼。參數 student：學生名字（從 list_homework 拿）。"""
    if student not in 作業們:
        return f"找不到「{student}」的作業"
    return 作業們[student][1]


@mcp.tool()
def send_email(to: str, subject: str, body: str) -> str:
    """寄信給學生（不可逆：寄出去就收不回來）。參數：收件人 email、主旨、內文。"""
    with 寄件匣.open("a", encoding="utf-8") as f:
        f.write(f"[{_dt.datetime.now():%H:%M:%S}] 寄給 {to}｜{subject}\n{body}\n\n")
    return f"（模擬）已寄給 {to}"


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--http", type=int, help="用 Streamable HTTP 開在這個 port（不給就用 stdio）")
    args = parser.parse_args()
    if args.http:
        mcp.settings.host = "127.0.0.1"
        mcp.settings.port = args.http
        mcp.run(transport="streamable-http")   # 網址：http://127.0.0.1:<port>/mcp
    else:
        mcp.run()


Writing /content/review-bot/deploy/homework_mcp.py


In [ ]:
#@title ▶ 執行：接上 MCP 插座，讓 agent 同時用 MCP 工具和本地工具
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # 準備、記下…（「0. 環境設定」寫好的小工具）
except ImportError:
    raise SystemExit("⚠️ 找不到環境：請先執行最上面的「0. 環境設定」那一格")
model = 準備()
from langchain.agents import create_agent
from langchain_mcp_adapters.client import MultiServerMCPClient
import agent

# 插座設定：用 stdio 在旁邊開一個「作業倉庫」程式，工具住在那個程式裡
插座 = MultiServerMCPClient({
    "作業倉庫": {"command": sys.executable, "args": [str(BASE / "deploy" / "homework_mcp.py")], "transport": "stdio"},
})
mcp工具 = await 插座.get_tools()          # Colab 可以直接寫 await
mcp名稱 = {t.name for t in mcp工具}

print("🧰 工具箱裡現在有：")
for t in agent.小助教的工具 + mcp工具:
    來源 = "MCP 插座（住在另一個程式）" if t.name in mcp名稱 else "本地函式（住在這個程式）"
    print(f"  - {t.name:<14} ← {來源}")
    print(f"      說明：{' '.join(t.description.split())[:70]}")
    print(f"      參數：{list(t.args)}")

# 最小權限：這個 agent 只需要「讀」作業，所以不把會寄信的 send_email 交給它
讀作業的工具 = [t for t in mcp工具 if t.name != "send_email"]
小助教 = create_agent(
    model, tools=agent.小助教的工具 + 讀作業的工具,
    system_prompt=agent.基本_system_prompt + "學生交的作業放在作業倉庫裡，要用 get_homework 拿到程式碼。",
)
問題 = "幫我看小美交的作業，跑跑看有沒有錯。"
print(f"\nUSER: {問題}\n")
順序, 步 = [], 0
# MCP 的工具只能用 async 呼叫，所以這裡用 astream（不是 1-B 前面的 stream）
async for event in 小助教.astream({"messages": [{"role": "user", "content": 問題}]}, stream_mode="updates"):
    for 節點, update in event.items():
        步 += 1
        for msg in (update or {}).get("messages", []):
            if getattr(msg, "tool_calls", None):
                for c in msg.tool_calls:
                    順序.append(c["name"])
                    來源 = "MCP 插座" if c["name"] in mcp名稱 else "本地函式"
                    print(f"--- 第 {步} 步 | 節點: {節點} ---  模型決定呼叫 -> {c['name']}({c['args']})  ← {來源}")
            elif msg.type == "tool":
                print(f"--- 第 {步} 步 | 節點: {節點} ---  [{msg.name} 的結果] {msg.text[:200]}")
            else:
                print(f"--- 第 {步} 步 | 節點: {節點} ---  [AIMessage] {msg.text[:300]}")
print("\n這次用工具的順序：", " → ".join(順序) or "（沒有用任何工具）")
記下("1-B 第 6 段：接上 MCP 插座", {"MCP 工具": sorted(mcp名稱), "用工具的順序": 順序})

### 👀 你看到了什麼

- 取得的清單有三把 MCP 工具；實際只交給 agent 兩把讀取工具。它們和 `run_code` 都有相同的基本介面：都有名字、說明書、參數，**都是「我可以呼叫的工具」**。
- 差別在**工具住在哪裡**：`get_homework` 真正執行的地方是旁邊那個「作業倉庫」程式，不是這個 notebook。
  其他 MCP server 可以在本機或遠端；認證方式依服務而定，MCP 不會自動替你取得權限。
- 檢查實際紀錄：是否先用 MCP 的 `get_homework` 取得作業，再用本地的 `run_code` 執行？這個例子讓你觀察**兩種來源的工具混在同一個迴圈裡用**。
- 插座上有 `send_email`，但我們**故意不把它交給這個 agent**：看作業不需要寄信，寄信又收不回來。只提供完成任務所需的工具與權限，這叫最小權限（第 2 堂 2-C 會再講）。
- 這裡的插座是 stdio：作業倉庫就跑在這台 Colab 上。遠端 MCP 可透過網路端點連線，2-E 練習 1 會把同一個 server 變成網址再接一次。

### 🤔 預測題（用量比較選讀）
三把本地工具的說明書大約幾個 token？接上作業倉庫的 3 個 MCP 工具呢？若換成下方仿照 GitHub 工具規模的教學清單，估算長度會差多少？

In [ ]:
#@title ▶ 執行：本地工具 vs MCP 插座，說明書佔掉多少 token
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # 準備、記下…（「0. 環境設定」寫好的小工具）
except ImportError:
    raise SystemExit("⚠️ 找不到環境：請先執行最上面的「0. 環境設定」那一格")
model = 準備()
import json
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain_mcp_adapters.client import MultiServerMCPClient
import agent

插座 = MultiServerMCPClient({
    "作業倉庫": {"command": sys.executable, "args": [str(BASE / "deploy" / "homework_mcp.py")], "transport": "stdio"},
})
mcp工具 = await 插座.get_tools()

# 示意用：規模仿 GitHub MCP server 的工具清單（名稱與說明是教學改寫，不是官方原文）
GitHub規模的工具 = [
    ('get_me', 'Get details of the authenticated GitHub user.', {}),
    ('search_repositories', 'Search GitHub repositories by keyword, language, stars or topic.', {'query': 'string', 'page': 'integer', 'per_page': 'integer'}),
    ('get_repository', 'Get metadata of a repository such as description, default branch and visibility.', {'owner': 'string', 'repo': 'string'}),
    ('create_repository', 'Create a new repository for the authenticated user or an organization.', {'name': 'string', 'description': 'string', 'private': 'boolean'}),
    ('fork_repository', 'Fork a repository to your account or an organization.', {'owner': 'string', 'repo': 'string', 'organization': 'string'}),
    ('list_branches', 'List branches in a repository with pagination.', {'owner': 'string', 'repo': 'string', 'page': 'integer'}),
    ('create_branch', 'Create a new branch from an existing branch or commit.', {'owner': 'string', 'repo': 'string', 'branch': 'string', 'from_branch': 'string'}),
    ('list_commits', 'List commits of a branch, optionally filtered by author or path.', {'owner': 'string', 'repo': 'string', 'sha': 'string', 'author': 'string'}),
    ('get_commit', 'Get a single commit including the diff and changed files.', {'owner': 'string', 'repo': 'string', 'sha': 'string'}),
    ('get_file_contents', 'Get the contents of a file or directory from a repository at a given ref.', {'owner': 'string', 'repo': 'string', 'path': 'string', 'ref': 'string'}),
    ('create_or_update_file', 'Create or update a single file in a repository with a commit message.', {'owner': 'string', 'repo': 'string', 'path': 'string', 'content': 'string', 'message': 'string', 'branch': 'string', 'sha': 'string'}),
    ('delete_file', 'Delete a file from a repository with a commit message.', {'owner': 'string', 'repo': 'string', 'path': 'string', 'message': 'string', 'branch': 'string'}),
    ('push_files', 'Push multiple files to a branch in a single commit.', {'owner': 'string', 'repo': 'string', 'branch': 'string', 'files': 'array', 'message': 'string'}),
    ('search_code', 'Search code across GitHub repositories using code search syntax.', {'query': 'string', 'page': 'integer', 'per_page': 'integer'}),
    ('list_issues', 'List issues in a repository filtered by state, labels, assignee or date.', {'owner': 'string', 'repo': 'string', 'state': 'string', 'labels': 'array', 'since': 'string'}),
    ('get_issue', 'Get the details of a specific issue including body and labels.', {'owner': 'string', 'repo': 'string', 'issue_number': 'integer'}),
    ('create_issue', 'Create a new issue with title, body, labels and assignees.', {'owner': 'string', 'repo': 'string', 'title': 'string', 'body': 'string', 'labels': 'array', 'assignees': 'array'}),
    ('update_issue', 'Update the title, body, state, labels or assignees of an issue.', {'owner': 'string', 'repo': 'string', 'issue_number': 'integer', 'title': 'string', 'body': 'string', 'state': 'string'}),
    ('add_issue_comment', 'Add a comment to an issue or pull request.', {'owner': 'string', 'repo': 'string', 'issue_number': 'integer', 'body': 'string'}),
    ('list_issue_comments', 'List comments on an issue with pagination.', {'owner': 'string', 'repo': 'string', 'issue_number': 'integer', 'page': 'integer'}),
    ('search_issues', 'Search issues and pull requests across repositories.', {'query': 'string', 'sort': 'string', 'order': 'string'}),
    ('list_pull_requests', 'List pull requests filtered by state, base or head branch.', {'owner': 'string', 'repo': 'string', 'state': 'string', 'base': 'string', 'head': 'string'}),
    ('get_pull_request', 'Get details of a pull request including mergeability and reviewers.', {'owner': 'string', 'repo': 'string', 'pull_number': 'integer'}),
    ('create_pull_request', 'Open a new pull request from a head branch into a base branch.', {'owner': 'string', 'repo': 'string', 'title': 'string', 'head': 'string', 'base': 'string', 'body': 'string', 'draft': 'boolean'}),
    ('update_pull_request', 'Update the title, body, base branch or state of a pull request.', {'owner': 'string', 'repo': 'string', 'pull_number': 'integer', 'title': 'string', 'body': 'string'}),
    ('merge_pull_request', 'Merge a pull request using merge, squash or rebase.', {'owner': 'string', 'repo': 'string', 'pull_number': 'integer', 'merge_method': 'string'}),
    ('get_pull_request_files', 'List files changed in a pull request with patch snippets.', {'owner': 'string', 'repo': 'string', 'pull_number': 'integer'}),
    ('get_pull_request_diff', 'Get the unified diff of a pull request.', {'owner': 'string', 'repo': 'string', 'pull_number': 'integer'}),
    ('create_pull_request_review', 'Create a review on a pull request with comments and a verdict.', {'owner': 'string', 'repo': 'string', 'pull_number': 'integer', 'body': 'string', 'event': 'string', 'comments': 'array'}),
    ('request_reviewers', 'Request reviewers for a pull request.', {'owner': 'string', 'repo': 'string', 'pull_number': 'integer', 'reviewers': 'array'}),
    ('list_releases', 'List releases of a repository.', {'owner': 'string', 'repo': 'string', 'page': 'integer'}),
    ('get_latest_release', 'Get the latest published release of a repository.', {'owner': 'string', 'repo': 'string'}),
    ('list_tags', 'List git tags of a repository.', {'owner': 'string', 'repo': 'string'}),
    ('list_workflows', 'List GitHub Actions workflows defined in a repository.', {'owner': 'string', 'repo': 'string'}),
    ('run_workflow', 'Trigger a workflow_dispatch run of a GitHub Actions workflow.', {'owner': 'string', 'repo': 'string', 'workflow_id': 'string', 'ref': 'string', 'inputs': 'object'}),
    ('get_workflow_run', 'Get the status and conclusion of a workflow run.', {'owner': 'string', 'repo': 'string', 'run_id': 'integer'}),
    ('get_job_logs', 'Download logs of jobs in a workflow run, optionally only failed jobs.', {'owner': 'string', 'repo': 'string', 'run_id': 'integer', 'failed_only': 'boolean'}),
    ('list_notifications', 'List notifications for the authenticated user.', {'all': 'boolean', 'participating': 'boolean', 'since': 'string'}),
    ('list_code_scanning_alerts', 'List code scanning alerts of a repository by state and severity.', {'owner': 'string', 'repo': 'string', 'state': 'string', 'severity': 'string'}),
    ('list_secret_scanning_alerts', 'List secret scanning alerts detected in a repository.', {'owner': 'string', 'repo': 'string', 'state': 'string'}),
]

def 轉成說明書(name, desc, params):
    props = {k: {"type": v, "description": k} for k, v in params.items()}
    return {"type": "function", "function": {"name": name, "description": desc,
            "parameters": {"type": "object", "properties": props, "required": list(params)[:2]}}}

def 估算說明書(說明書們):
    return 估算_token(json.dumps(說明書們, ensure_ascii=False))

三組 = [
    ("本地工具（自己寫的）", [convert_to_openai_tool(t) for t in agent.小助教的工具]),
    ("作業倉庫 MCP（真的接上）", [convert_to_openai_tool(t) for t in mcp工具]),
    ("GitHub 規模的 MCP（示意）", [轉成說明書(*t) for t in GitHub規模的工具]),
]
結果 = {}
for 名稱, 說明書們 in 三組:
    結果[名稱] = 估算說明書(說明書們)
    print(f"{名稱:<18} {len(說明書們):>3} 個工具：約 {結果[名稱]:>5,} token  " + "█" * max(1, 結果[名稱] // 100))
本地 = 結果["本地工具（自己寫的）"]
大的 = 結果["GitHub 規模的 MCP（示意）"]
print(f"\n若把這份 GitHub 規模清單全部交給模型，一次呼叫的工具說明約有 {大的:,} token 的說明書，是本地工具的 {大的 / 本地:.1f} 倍。")
print("（token 數是用字數估算的，只看大小關係就好）")
記下("1-B 第 6 段：MCP 說明書的 token", 結果)

### 👀 你看到了什麼

- 作業倉庫只有 3 個工具，說明書不多；這個示意清單有 40 個，說明書明顯較長。實際成本取決於選了哪些工具、模型格式與 tokenizer，這裡只看估算的大小關係。
- 對 agent 來說，本地工具和 MCP 工具用起來一樣；差別在工具住哪、怎麼認證、說明書佔多少桌子。
- 第 2 堂會用 **Skill** 按需載入任務指引。Skill 不會自動移除工具定義；要縮減工具清單，仍須由程式篩選或延後提供。

---
## 第 7 段（課後選做）：用 LangGraph 看懂 agent 的核心迴圈

> 📍 **對應簡報**：投影片 47「附錄選做：模型與工具的核心迴圈」

核心練習已用過 `create_agent`。這段另用 LangGraph 示範「模型 ↔ 工具」迴圈，不是 `create_agent` 的完整實作；可課後再讀。

沿用 1-A 的三個零件：**State** 保存訊息、**Node** 執行工作、**Edge** 連接下一步；條件邊依工具要求選路：
一個問模型的節點 `ask_model`、一個跑工具的節點 `tools`、一條「有沒有要用工具」的 if（`tools_condition`），繞成一個圈。
程式在 `agent.py` 的 `建立_展開版_agent`。

| 元件 | 做什麼 |
| --- | --- |
| `@tool` | 函式 → 模型看得到的工具 |
| `bind_tools` | 工具說明書附在每次呼叫上 |
| `ask_model` | 問模型，回文字或 tool_calls |
| `ToolNode` | 讀 tool_calls、執行、包結果 |
| `tools_condition` | 內建路由：有 tool_calls → tools，沒有 → END |
| `add_edge("tools", "ask_model")` | 工具跑完回去問模型，形成循環 |

### 🤔 預測題
流程圖上會有幾個節點？哪一條是條件邊？

In [ ]:
#@title ▶ 執行：自己用 LangGraph 組出核心迴圈
作業 = "變數打錯字"  #@param ["變數打錯字", "正確的程式", "縮排亂掉", "少了右括號", "我的作業"]
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # 準備、記下…（「0. 環境設定」寫好的小工具）
except ImportError:
    raise SystemExit("⚠️ 找不到環境：請先執行最上面的「0. 環境設定」那一格")
model = 準備()
import agent

展開版 = agent.建立_展開版_agent(model)
顯示流程圖(展開版)
events = agent.pretty_trace(展開版, agent.作業訊息(取得作業(作業)))
記下(f"1-B 第 7 段：展開版 agent（{作業}）", events)

### 👀 你看到了什麼

- 不計 START／END，共兩個工作節點：`ask_model` 和 `tools`。從 `ask_model` 出來的是條件邊（虛線）：要用工具就去 `tools`，不用就結束。
- `tools` 跑完一定回到 `ask_model`，這條往回指的箭頭就是迴圈。
- 這堂課用 LangChain 的 `create_agent` 建立 agent；它的執行基礎是 LangGraph。

---
## 第 1 堂收尾

- **Agent ＝ 模型 ＋ 工具 ＋ 迴圈**；模型只會「說」要用哪個工具，做事的是你的程式。
- **流程安排不能省略的步驟，agent 處理需要彈性判斷的部分**；兩者可以套在一起（外面是固定流程、某一站裡面放一個 agent），就是課名的「雙拼」（簡報投影片 42，第 2 堂 2-D 會做出來）。
- 📝 **課後作業**：把你自己寫過的一段作業放進「我的作業」，分別在 1-A 和 1-B 跑一次，
  截圖 1-A 的 State 歷程與 1-B 的工具執行紀錄，寫一句話說哪邊的結果你比較信任、為什麼。

## 收尾：存檔與交作業

- 💾 **存到雲端硬碟**：把 `/content/review-bot/`（你改過的手冊、程式、State 紀錄）存一份到自己的 Google 雲端硬碟。
- ⬇️ **下載 zip**：直接下載到電腦。
- 📸 **最後一格**：印出實驗格存下的結果。挑選相關紀錄截圖，再依作業要求補上觀察與理由。

這些是實驗格透過 `記下()` 保存的結果，並非每次執行的所有中間狀態。
同一執行環境的紀錄會累積，可能包含其他 notebook 的實驗；請按輸出標題辨認。

瀏覽器暫時斷線不一定會清掉檔案；執行階段被刪除或回收時，`/content/` 的檔案才會失去。
想保留手冊、程式和 checkpoint，請先備份。notebook 本身則用上方選單「檔案 → 在雲端硬碟中儲存副本」，兩種存檔用途不同。
回來操作時先確認執行階段仍在；若已重設，重跑「0. 環境設定」及本段準備格，再依需要還原自己備份的檔案。

In [16]:
#@title 💾 存到 Google 雲端硬碟（會跳出授權視窗）
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # 準備、記下…（「0. 環境設定」寫好的小工具）
except ImportError:
    raise SystemExit("⚠️ 找不到環境：請先執行最上面的「0. 環境設定」那一格")
model = 準備()
存到雲端硬碟()

Mounted at /content/drive
✅ 已存到雲端硬碟：/content/drive/MyDrive/langgraph-skills-workshop/20261102-083654
   （notebook 本身請用上方選單「檔案 → 在雲端硬碟中儲存副本」）


In [17]:
#@title ⬇️ 下載 zip
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # 準備、記下…（「0. 環境設定」寫好的小工具）
except ImportError:
    raise SystemExit("⚠️ 找不到環境：請先執行最上面的「0. 環境設定」那一格")
model = 準備()
下載壓縮檔()

✅ 已打包：/content/review-bot-20261102-083655.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
#@title 📸 印出已記錄的實驗結果（選取相關紀錄交作業）
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # 準備、記下…（「0. 環境設定」寫好的小工具）
except ImportError:
    raise SystemExit("⚠️ 找不到環境：請先執行最上面的「0. 環境設定」那一格")
model = 準備()
印出歷程()